In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# File Paths
IN_FP  = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Table Generation/bundesliga_std.csv"
OUT_FP = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Table Generation/bundesliga_standings_all_seasons.csv"

In [4]:
#Loading data, and using necessary columns

df = pd.read_csv(IN_FP, dtype=str, low_memory=False)

need = ["season","home_team","away_team","hometeamgoals","awayteamgoals"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
df["hometeamgoals"] = pd.to_numeric(df["hometeamgoals"], errors="coerce")
df["awayteamgoals"] = pd.to_numeric(df["awayteamgoals"], errors="coerce")

use = df.dropna(subset=["season","hometeamgoals","awayteamgoals"]).copy()

In [5]:
# --- Comparing goals scored to compute results ---

hg = use["hometeamgoals"].astype(int)
ag = use["awayteamgoals"].astype(int)

home_win  = (hg > ag).astype(int)
draw      = (hg == ag).astype(int)
home_loss = (hg < ag).astype(int)

In [6]:
# --- Building per-team rows (home & away) for ALL seasons ---

home = pd.DataFrame({
    "season": use["season"].astype(int),
    "team":   use["home_team"],
    "points": (home_win*3 + draw).astype(int),
    "wins":   home_win,
    "draws":  draw,
    "losses": home_loss,
    "gf":     hg,
    "ga":     ag,
    "home_games": 1,
    "away_games": 0,
})

away = pd.DataFrame({
    "season": use["season"].astype(int),
    "team":   use["away_team"],
    "points": (home_loss*3 + draw).astype(int), # away gets 3 points when home team loses
    "wins":   home_loss,
    "draws":  draw,
    "losses": home_win,
    "gf":     ag,
    "ga":     hg,
    "home_games": 0,
    "away_games": 1,
})

long = pd.concat([home, away], ignore_index=True)

In [7]:
# --- Combined to end-of-season table for each (season, team) ---

agg = long.groupby(["season","team"], as_index=False).sum(numeric_only=True)
agg["gd"] = agg["gf"] - agg["ga"]


In [8]:
# --- Rank within each season: Points ↓, GD ↓, GF ↓, Team ↑ ---

agg = agg.sort_values(
    ["season","points","gd","gf","team"],
    ascending=[True, False, False, False, True],
    kind="mergesort",
)
agg["rank"] = agg.groupby("season").cumcount() + 1


In [9]:
# --- Final view (only required columns) ---

standings_all = (
    agg.rename(columns={"team":"team_name"})[
        ["season","team_name","points","wins","draws","losses","gf","ga","gd","rank"]
    ]
    .sort_values(["season","rank"])
    .reset_index(drop=True)
)


In [12]:
# integrity check

check = (
    agg.assign(games=agg["home_games"] + agg["away_games"])
       .groupby("season")
       .agg(
           teams=("team", "nunique"),
           total_games=("games", "sum"),
           min_games=("games", "min"),
           max_games=("games", "max"),
       )
       .assign(
           expected_teams=18,
           expected_total_games=lambda d: d["expected_teams"]*(d["expected_teams"]-1),  # 18*(18-1)=306
           expected_games_per_team=34,
           teams_ok=lambda d: d["teams"].eq(d["expected_teams"]),
           matches_ok=lambda d: d["total_games"].eq(d["expected_total_games"]),
           per_team_ok=lambda d: (d["min_games"].eq(d["expected_games_per_team"]) &
                                  d["max_games"].eq(d["expected_games_per_team"])),
       )
       .reset_index()
)

display(check)

,season,teams,total_games,min_games,max_games,expected_teams,expected_total_games,expected_games_per_team,teams_ok,matches_ok,per_team_ok
0,2004,18,612,34,34,18,306,34,True,False,True
1,2005,18,612,34,34,18,306,34,True,False,True
2,2006,18,612,34,34,18,306,34,True,False,True
3,2007,18,612,34,34,18,306,34,True,False,True
4,2008,18,612,34,34,18,306,34,True,False,True
5,2009,18,612,34,34,18,306,34,True,False,True
6,2010,18,612,34,34,18,306,34,True,False,True
7,2011,18,612,34,34,18,306,34,True,False,True
8,2012,18,612,34,34,18,306,34,True,False,True
9,2013,18,612,34,34,18,306,34,True,False,True


In [13]:
display(standings_all[standings_all["season"] == 2024].head(20))

,season,team_name,points,wins,draws,losses,gf,ga,gd,rank
360,2024,BAY,82,25,7,2,99,32,67,1
361,2024,LEVK,69,19,12,3,72,43,29,2
362,2024,EINF,60,17,9,8,68,46,22,3
363,2024,DOR,57,17,6,11,71,51,20,4
364,2024,FRE,55,16,7,11,49,53,-4,5
365,2024,MAI,52,14,10,10,55,43,12,6
366,2024,RBL,51,13,12,9,53,48,5,7
367,2024,WERB,51,14,9,11,54,57,-3,8
368,2024,STU,50,14,8,12,64,53,11,9
369,2024,M'G,45,13,6,15,55,57,-2,10


In [14]:
Path(OUT_FP).parent.mkdir(parents=True, exist_ok=True)
standings_all.to_csv(OUT_FP, index=False)